# 1. Create a Track

In **Rolland**, building a track model follows a strict object-oriented workflow. First, the individual physical components of the track (such as the rail, pads, sleepers, and ballast) must be created as independent instances. After these components are defined with their respective geometric, elastic, and damping properties, they are passed into a track class to be assembled into a complete `Track` model.

(track-model)=
![Rail model and eccentricities.](../images/eccentricities.png)

## 1.1 Simplified Assumptions & Theoretical Background

The mechanical track model is fundamentally based on a time-domain finite-difference approach. The track is modeled as a **single-sided track** (single-rail model). This implies that we simulate one rail and the corresponding half of the sleeper and track bed.

Key simplifying assumptions include:

* **Support Idealization**: The rail support is idealized as a single point on the rail foot, defined by a vertical and a lateral offset ($z_f$ and $y_f$) relative to the rail centroid $C_r$. The thickness of the rail pad is considered negligible.
* **Equivalent Rigid Sleeper**: For a two-layer track (where sleeper displacements are permitted), an equivalent rigid sleeper model is adopted. Since we use a single-rail model, vertical rail displacements induce a bending moment due to the asymmetric support. Therefore, the sleeper centroid $C_s'$ (originally midway between rails) is reduced to a point $C_s$ on the rail symmetry axis. Consequently, the effective vertical sleeper mass and ballast stiffness diminish to approximately half (for a standard 1435 mm track gauge).
* **Multiple Degrees of Freedom (DOFs)**: The numerical model evaluates 7 DOFs for the rail at its centroid (vertical and lateral bending, torsion, longitudinal, and warping). Excitation forces applied at the rail head are kinematically transformed to the centroid. If sleeper motion is permitted, it introduces 6 additional DOFs (translation and rotation).
* **Damping**: Viscous damping is utilized instead of hysteretic damping to allow for robust time-domain integration. Damping coefficients are carefully tuned to corresponding cut-on frequencies.


## 1.2 Defining a Rail

The fundamental component of the track is the rail. We can import standard pre-defined rail geometries directly from the database. Here, we load the standard `60E1` (formerly UIC60) rail profile.


In [1]:
from rolland.database.rail.db_rail import rail_60E1, available_rails

# Print all available rail profiles in the database
print("Available rail profiles:", available_rails())

# Load the standard 60E1 rail profile
rail = rail_60E1  # Instantiate the rail using the predefined profile

Available rail profiles: ['rail_49E1', 'rail_49E5', 'rail_54E1', 'rail_54E3', 'rail_54E4', 'rail_60E1', 'rail_60E2']


## 1.3 Defining a Sleeper and Slab

Sleepers and slabs support the rails. To define a sleeper, we need its physical dimensions (length, width, height) and material density (e.g., concrete). 

**Sleeper Assumptions & Calculations:**
*   The sleeper is assumed to be a uniform rectangular cuboid.
*   The total mass (`m`) is computed as `density * volume`. However, remember the **equivalent sleeper model**: the track instantiation will internally apply a transformation matrix to halve the effective mass and stiffness for the single-rail model.
*   The moments of inertia (`I_sz`, `I_sy`, `I_sx`) are calculated using standard formulas for a rectangular block, and are normalized by the density as required by the model parameters.
*   To convert discrete sleeper properties into a continuous `Slab` model (for continuous support), the mass and inertia values are "smeared" by dividing them by the sleeper distance (`slep_dist`).
*   The top (`z_st`) and bottom (`z_sb`) coordinates are set relative to the sleeper center of mass.


In [2]:
from rolland import Sleeper, Slab

slep_dist = 0.6

# Sleeper geometry and material
sl_len = 2.5
sl_width = 0.245
sl_height = 0.185

rhos = 2648
m = rhos * sl_len * sl_width * sl_height

# Moments of inertia (normalized by density for the model input)
I_sz = (sl_len**2 + sl_width**2) * m / 12 * (1 / rhos)
I_sy = (sl_height**2 + sl_width**2) * m / 12 * (1 / rhos)
I_sx = (sl_len**2 + sl_height**2) * m / 12 * (1 / rhos)

# Define discrete Sleeper
sleeper = Sleeper(
    ms=m,  # Mass
    rhos=rhos,  # Density
    Is_x=I_sx,  # Inertia X
    Is_y=I_sy,  # Inertia Y
    Is_z=I_sz,  # Inertia Z
    lengs=sl_len,  # Length
    wdths=sl_width,  # Width
    heights=sl_height,  # Height
    z_st=-sl_height / 2,  # Top Z
    z_sb=sl_height / 2,  # Bottom Z
)

# Define equivalent continuous Slab
slab = Slab(
    ms=m / slep_dist,  # Smeared mass
    Is_x=I_sx / slep_dist,  # Smeared inertia X
    Is_y=I_sy / slep_dist,  # Smeared inertia Y
    Is_z=I_sz / slep_dist,  # Smeared inertia Z
    lengs=sl_len,  # Length
    rhos=rhos,  # Density
    equ_wdths=sl_width,  # Width
    heights=sl_height,  # Height
    z_st=-sl_height / 2,  # Top Z
    z_sb=sl_height / 2,  # Bottom Z
)

## 1.4 Defining Pads and Ballast

The track bed incorporates elastic and damping components—specifically the rail pads (between the rail and sleeper) and the ballast (beneath the sleeper).

**Pad and Ballast Assumptions:**
*   Properties like stiffness (`sp`, `sb`) and loss factors (`etap`, `etab`) are defined for vertical (`z`), lateral (`y`), and longitudinal (`x`) degrees of freedom.
*   Rotational and warping stiffnesses are derived internally from the translational stiffness and the geometry of the pad.
*   Just like with the sleeper, continuous equivalents (`ContPad` and continuous `Ballast`) are created by dividing the discrete stiffness properties by the sleeper distance (`slep_dist`).

![Elastic and damping properties corresponding to the rail pad and the ballast for each degree of freedom (warping is not shown).](../images/mech_props_pad_ballast.png)


In [ ]:
from rolland import DiscrPad, ContPad, Ballast

# Discrete Pad
discrpad = DiscrPad(
    sp_z=120e6,  # Vert. stiffness
    sp_y=40e6,  # Lat. stiffness
    sp_x=40e6,  # Long. stiffness
    etap_z=0.2,  # Vert. loss
    etap_y=0.2,  # Lat. loss
    etap_x=0.2,  # Long. loss
    etap_r=0.2,  # Rot. loss
    wdthp=0.15  # Pad width
)

# Continuous Pad
contpad = ContPad(
    sp_z=120e6 / slep_dist,  # Smeared vert. stiffness
    sp_y=40e6 / slep_dist,  # Smeared lat. stiffness
    sp_x=40e6 / slep_dist,  # Smeared long. stiffness
    etap_z=0.2,  # Vert. loss
    etap_y=0.2,  # Lat. loss
    etap_x=0.2,  # Long. loss
    etap_r=0.2,  # Rot. loss
    wdthp=0.15  # Pad width
)

# Discrete Ballast
discrballast = Ballast(
    sb_z=120e6,  # Vert. stiffness
    sb_y=120e6,  # Lat. stiffness
    sb_x=120e6,  # Long. stiffness
    etab_z=1,  # Vert. loss
    etab_y=2,  # Lat. loss
    etab_x=2,  # Long. loss
    etab_r=2,  # Rot. loss
)

# Continuous Ballast
contballast = Ballast(
    sb_z=120e6 / slep_dist,  # Smeared vert. stiffness
    sb_y=120e6 / slep_dist,  # Smeared lat. stiffness
    sb_x=120e6 / slep_dist,  # Smeared long. stiffness
    etab_z=1,  # Vert. loss
    etab_y=2,  # Lat. loss
    etab_x=2,  # Long. loss
    etab_r=2,  # Rot. loss
)

## 1.5 Assemble the Track

Now that all components (both discrete and continuous) are fully defined, we assemble them into a specific `Track` instance. 

### Track Design Possibilities in Rolland

The API offers a highly flexible set of track classes, catering to different modeling approaches (1-layer vs. 2-layer) and support types (continuous vs. discrete). Furthermore, the API differentiates between rigid slab tracks and ballasted tracks.

**1. Slab Tracks (Rigid Support)**
These tracks represent configurations where the rail is supported by a rigid foundation (e.g., direct fastening on a concrete slab):
* `ContSlabSingleRailTrack`: Continuous rail, continuous pads, and a continuous rigid slab.
* `SimplePeriodicSlabSingleRailTrack`: Continuous rail, discrete pads, and discrete rigid support points at uniform intervals.
* `ArrangedSlabSingleRailTrack`: Similar to the simple periodic track, but allows for structural irregularities. Mounting properties (pad stiffness, spacing) can vary periodically or stochastically along the track using `PeriodicArrangement` or `RandomArrangement`.

**2. Ballasted Tracks (Elastic Support)**
These tracks represent classic configurations with sleepers embedded in a resilient ballast layer:
* `ContBallastedSingleRailTrack`: Continuous rail, pads, sleeper layer, and ballast layer. All properties are continuous (per meter).
* `SimplePeriodicBallastedSingleRailTrack`: Standard ballasted track. Continuous rail, discrete pads, discrete sleepers, and discrete ballast support at uniform sleeper spacing.
* `ArrangedBallastedSingleRailTrack`: Allows variations in the discrete pad, sleeper, and ballast properties along the track (e.g., varying sleeper spacing or localized degraded ballast stiffness).

### Defaults and Internal Calculations
When instantiating components, most structural properties (mass, stiffness) do not have arbitrary defaults and must be explicitly provided by the user. However, some important automated behaviors and default values are present:
* **Equivalent Sleeper Model (`y_sc = 0.7175 m`)**: By default, the `Sleeper` and `Slab` components have a lateral eccentricity parameter `y_sc` set to 0.7175 m (half of the standard 1435 mm gauge). The parameter `equi_sm` is `True` by default, activating Kostovasilis' equivalent rigid sleeper formulation that automatically scales down vertical mass and stiffness to account for single-rail bending.
* **Rotational Stiffness and Damping**: Rotational properties of pads and ballast (e.g., `sp_xr`, `sp_yr`, `sp_zr`) are derived automatically from the translational stiffnesses (`sp_x`, `sp_y`, `sp_z`) and the component's geometric dimensions (e.g., `wdthp`, `lengs`, `wdths`).
* **Damping Conversion**: You specify dimensionless loss factors (e.g., `etap_z`, `etab_z`). The underlying numerical solver (e.g., the finite-difference time-domain solver) will automatically compute the corresponding viscous damping coefficients ($d = \eta s / \omega_d$) tailored to the dominant cut-on or pinned-pinned resonance frequencies.

Here we instantiate a **2-layer discrete ballasted track** with uniform support using `SimplePeriodicBallastedSingleRailTrack`. During instantiation, the components' mechanical properties are combined, eccentricities are evaluated, and the transformation to the rail centroid is completed.


In [4]:
from rolland.track import SimplePeriodicBallastedSingleRailTrack
from rolland.domainsetup import DomSetup
from rolland.boundary import CFSPML

# Assemble a discrete ballasted track
track = SimplePeriodicBallastedSingleRailTrack(
    rail=rail,  # Rail
    pad=discrpad,  # Pads
    sleeper=sleeper,  # Sleepers
    ballast=discrballast,  # Ballast
    distance=slep_dist,  # Spacing
    z_f=0.081,  # Z-eccentricity
    y_f=0.0,  # Y-eccentricity
)

Track assembled! Grid points nx: 1188


## 1.6 Setup the Domain

After creating the track, we need to embed it into a computational domain (`DomSetup`). Since railway tracks are physically continuous over vast distances, simulating them in a finite numerical model requires special boundary treatments to avoid unphysical wave reflections at the domain ends.

### Applying the CFS-PML Boundary
To achieve this, Rolland utilizes a **Complex Frequency-Shifted Perfectly Matched Layer (CFS-PML)**. The PML acts as an absorbing boundary zone added to both ends of the internal track domain. When waves enter the PML, they are gradually dissipated.

* **Default Parameters:** The CFS-PML mathematical formulation involves several damping parameters (like damping exponent `m`, max damping coefficient `a`, and frequency-shifting factor `alpha`). In Rolland, you can safely use the default values for all these internal parameters. They have been empirically optimized and generalized to provide highly robust absorption for standard rail types and complex multi-DOF track configurations. There is no need for case-by-case recalibration.
* **Conservative Boundary Length (`l_bound = 10.0`):** The most critical parameter you control is the physical length of the absorbing boundary domain (`l_bound`). A length of **10 meters** is generally considered a good, conservative value. Because the track exhibits complex, highly dispersive flexural waves with varying wavelengths across different frequencies, providing a spatial buffer of 10 meters ensures sufficient physical distance for the PML damping profile to fully attenuate all wave types before they can reflect back into the main interior domain.

(domain)=
![Interior and boundary domain.](../images/domain.png)


In [ ]:
# Setup the domain
domain = DomSetup(
    track=track,  # Track
    bound=CFSPML(l_bound=10.0)  # PML length
)

print(f"Track assembled! Grid points nx: {domain.nx}")
